# EDA - Funil SaaS Conta Azul

Este notebook demonstra a investigacao analitica do case de Business Analytics. Ele complementa o dashboard Streamlit, mostrando o raciocinio por tras dos principais achados.

Objetivos:

- Ler o CSV original com Pandas.
- Fazer profiling da base.
- Validar qualidade e consistencia dos dados.
- Calcular o funil geral.
- Analisar conversao por canal, dispositivo, pais e mes.
- Analisar NPS geral, compradores e nao compradores.
- Gerar graficos com Plotly.
- Consolidar conclusoes antes do dashboard.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
import plotly.express as px
import plotly.io as pio

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == 'notebooks':
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


DATA_FILE = ROOT_DIR / 'saas_funnel_case_10k_refresh_(4)_(2).csv'

# Paleta inspirada na identidade visual da Conta Azul.
CONTA_AZUL_COLORS = [
    '#1B69C8',  # azul principal
    '#14A7E0',  # azul claro
    '#00B8D9',  # ciano
    '#49C96D',  # verde de acao/positivo
    '#FDB913',  # amarelo de destaque
    '#0B2F6B',  # azul escuro
]
px.defaults.color_discrete_sequence = CONTA_AZUL_COLORS
px.defaults.template = 'plotly_white'
pio.renderers.default = 'vscode'
print(f"Arquivo CSV encontrado em: {DATA_FILE}")
print(f"Arquivo existe? {DATA_FILE.exists()}")

## 1. Leitura do CSV com Pandas

In [ ]:
raw = pd.read_csv(DATA_FILE)
raw.head()

In [ ]:
pd.DataFrame({
    "metrica": ["linhas", "colunas"],
    "valor": [raw.shape[0], raw.shape[1]]
})

## 2. Tratamento inicial e campos derivados

A base tem grao de uma linha por usuario visitante. Criamos campos derivados para facilitar a leitura de funil, tempo e NPS.

In [ ]:
df = raw.copy()
df['dt_visit'] = pd.to_datetime(df['dt_visit'], errors='coerce')
df['days_to_signup'] = df['days_to_signup'].astype('Int64')
df['days_to_purchase'] = df['days_to_purchase'].astype('Int64')
df['nps_score'] = df['nps_score'].astype('Float64')

df['visit_month'] = df['dt_visit'].dt.to_period('M').astype(str)
df['signup_date'] = df['dt_visit'] + pd.to_timedelta(df['days_to_signup'], unit='D')
df['purchase_date'] = df['signup_date'] + pd.to_timedelta(df['days_to_purchase'], unit='D')

df['funnel_stage'] = 'Visit only'
df.loc[(df['signup'] == 1) & (df['purchase'] == 0), 'funnel_stage'] = 'Signup only'
df.loc[df['purchase'] == 1, 'funnel_stage'] = 'Purchased'

df['nps_class'] = 'No response'
df.loc[df['nps_score'].notna() & (df['nps_score'] < 7), 'nps_class'] = 'Detractor'
df.loc[df['nps_score'].notna() & (df['nps_score'] >= 7) & (df['nps_score'] < 9), 'nps_class'] = 'Passive'
df.loc[df['nps_score'].notna() & (df['nps_score'] >= 9), 'nps_class'] = 'Promoter'

df.head()

## 3. Profiling da base

In [ ]:
profile = pd.DataFrame({
    'metrica': [
        'linhas', 'colunas', 'usuarios_unicos', 'usuarios_duplicados',
        'data_minima_visita', 'data_maxima_visita', 'canais', 'dispositivos', 'paises', 'planos'
    ],
    'valor': [
        len(df), df.shape[1], df['user_id'].nunique(), df['user_id'].duplicated().sum(),
        df['dt_visit'].min().date(), df['dt_visit'].max().date(),
        df['channel'].nunique(), df['device'].nunique(), df['country'].nunique(), df['plan'].nunique(dropna=True)
    ]
})
profile

In [ ]:
df[['channel', 'device', 'country', 'plan']].describe(include='all')

## 4. Validacoes de qualidade

As validacoes abaixo garantem que as metricas de funil nao estao contaminadas por inconsistencias basicas.

In [ ]:
checks = pd.DataFrame([
    ('purchase_sem_signup', ((df['purchase'] == 1) & (df['signup'] == 0)).sum()),
    ('plan_sem_purchase', (df['plan'].notna() & (df['purchase'] == 0)).sum()),
    ('purchase_sem_plan', ((df['purchase'] == 1) & df['plan'].isna()).sum()),
    ('days_signup_sem_signup', (df['days_to_signup'].notna() & (df['signup'] == 0)).sum()),
    ('signup_sem_days_signup', ((df['signup'] == 1) & df['days_to_signup'].isna()).sum()),
    ('days_purchase_sem_purchase', (df['days_to_purchase'].notna() & (df['purchase'] == 0)).sum()),
    ('purchase_sem_days_purchase', ((df['purchase'] == 1) & df['days_to_purchase'].isna()).sum()),
    ('nps_score_sem_resposta', (df['nps_score'].notna() & (df['respondeu_nps'] == 0)).sum()),
    ('resposta_sem_nps_score', ((df['respondeu_nps'] == 1) & df['nps_score'].isna()).sum()),
    ('nps_fora_intervalo', ((df['nps_score'] < 0) | (df['nps_score'] > 10)).sum()),
    ('respostas_nps_nao_compradores', ((df['respondeu_nps'] == 1) & (df['purchase'] == 0)).sum()),
], columns=['regra', 'resultado'])

checks

## 5. DuckDB: SQL local e views analiticas

A partir daqui usamos DuckDB para calcular as principais metricas em SQL, mantendo a analise reproduzivel.

In [ ]:
con = duckdb.connect(database=':memory:')
con.register('stg_funnel_users', df)

create_views_sql = (ROOT_DIR / 'sql' / '01_create_views.sql').read_text(encoding='utf-8')
con.execute(create_views_sql)

con.execute("select table_name from information_schema.tables order by table_name").df()

## 6. Funil geral

In [ ]:
overall = con.execute('select * from vw_funnel_overall').df()
overall

In [ ]:
funnel_steps = pd.DataFrame({
    'etapa': ['Visitas', 'Signups', 'Compras'],
    'usuarios': [int(overall.loc[0, 'visits']), int(overall.loc[0, 'signups']), int(overall.loc[0, 'purchases'])]
})

fig = px.funnel(funnel_steps, x='usuarios', y='etapa', title='Funil geral - visitas, signups e compras',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.show()

Leitura: o funil apresenta 10.000 visitas, 2.983 signups e 919 compras. O maior gargalo absoluto esta antes do signup, seguido pela perda pos-signup.

## 7. Analise por canal

In [ ]:
channel = con.execute('select * from vw_funnel_by_channel order by visit_to_purchase_rate desc').df()
channel

In [ ]:
fig = px.bar(
    channel.sort_values('visit_to_purchase_rate'),
    x='visit_to_purchase_rate',
    y='channel',
    orientation='h',
    text='purchases',
    title='Visit to purchase por canal',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_xaxes(tickformat='.0%')
fig.show()

Leitura: `referral` e `email` sao os canais mais eficientes. `organic` tem forte volume absoluto. `paid` e `social` precisam de otimizacao antes de escala.

## 8. Analise por dispositivo

In [ ]:
device = con.execute('select * from vw_funnel_by_device order by visit_to_purchase_rate desc').df()
device

In [ ]:
fig = px.bar(
    device,
    x='device',
    y=['visit_to_signup_rate', 'visit_to_purchase_rate', 'signup_to_purchase_rate'],
    barmode='group',
    title='Taxas de conversao por dispositivo',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_yaxes(tickformat='.0%')
fig.show()

Leitura: desktop converte melhor que mobile. Como mobile concentra mais visitas, melhorar esse fluxo pode gerar impacto relevante.

## 9. Analise por pais e por mes

In [ ]:
country = con.execute('select * from vw_funnel_by_country order by visits desc').df()
country

In [ ]:
monthly = con.execute('select * from vw_funnel_by_month order by visit_month').df()
monthly

In [ ]:
fig = px.line(
    monthly,
    x='visit_month',
    y=['visit_to_signup_rate', 'visit_to_purchase_rate', 'signup_to_purchase_rate'],
    markers=True,
    title='Evolucao mensal das taxas de conversao',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_yaxes(tickformat='.0%')
fig.show()

Leitura: as taxas por pais sao relativamente proximas. Setembro apresenta a melhor taxa de compra sobre visitas e melhor conversao pos-signup.

## 10. Analise de NPS

In [ ]:
nps_summary = con.execute("""
select * from vw_nps_summary
union all
select
    'Geral' as segmento,
    count(*) as respostas,
    avg(nps_score) as nps_medio,
    sum(case when nps_class = 'Promoter' then 1 else 0 end) as promotores,
    sum(case when nps_class = 'Passive' then 1 else 0 end) as passivos,
    sum(case when nps_class = 'Detractor' then 1 else 0 end) as detratores,
    (
        sum(case when nps_class = 'Promoter' then 1 else 0 end)
        - sum(case when nps_class = 'Detractor' then 1 else 0 end)
    ) * 100.0 / count(*) as nps
from stg_funnel_users
where nps_score is not null
""").df()
nps_summary

In [ ]:
fig = px.bar(
    nps_summary,
    x='segmento',
    y='nps',
    text='nps',
    title='NPS geral, compradores e nao compradores',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_traces(texttemplate='%{text:.1f}')
fig.show()

In [ ]:
nps_channel = con.execute('select * from vw_nps_by_channel order by nps desc').df()
nps_channel

Leitura: o NPS geral esconde duas realidades. Compradores possuem NPS positivo e alto, enquanto nao compradores possuem NPS negativo, indicando friccao ou desalinhamento de valor antes da compra.

## 11. Principais conclusoes antes do dashboard

1. O funil tem dois gargalos relevantes: visita para signup e signup para compra.
2. `referral` e o canal de maior qualidade e deveria ser priorizado em experimentos de crescimento.
3. `paid` tem volume, mas baixa eficiencia; a recomendacao e otimizar antes de escalar investimento.
4. Mobile tem alto volume e baixa conversao relativa, indicando oportunidade de UX, formulario, onboarding ou checkout.
5. `organic` combina escala e eficiencia, sendo importante para sustentacao de aquisicao.
6. O NPS deve ser analisado separando compradores e nao compradores; a media geral mascara a insatisfacao dos nao compradores.
7. As recomendacoes de negocio devem focar em referral, paid, mobile, organic, email/lifecycle e pesquisa qualitativa com nao compradores.